## Manual annotation using gene expression

In [ ]:
# eval "$(conda shell.bash hook)"
# conda init
# conda activate /work/islet_cartography_scrna/scrna_cartography_py_analysis
# python -m ipykernel install --user --name scrna_cartography_py_analysis --display-name "py_analysis"

In [1]:
# Path and system utilities
import os                    # Operating system interface
import sys                   # System-specific parameters and functions
import glob                  # File pattern matching
from pathlib import Path     # Object-oriented filesystem paths
from pyhere import here      # Reproducible project paths

# Single-cell data handling
import anndata as ad            # Core data structure for single-cell data
import scanpy as sc          # Analysis and visualization of single-cell data
import pyucell as uc         # Module score

# dataframes
import pandas as pd
import numpy as np
# Micellaneous utilities
import warnings              # Suppress or manage warnings

# Custom modules and functions
sys.path.append(str(here('scripts/misc')))  # Add custom script path to system
import my_anndata as ma                    # Custom AnnData utilities
import misc as mi
import diff_genes as dg

In [2]:
# Paths
base_dir = str(here('data/annotate/'))
plot_dir = os.path.join(base_dir, 'plot') 
files_dir = os.path.join(base_dir, 'files') 

anndata_dir = str(here('data/anndata/'))
harmo_dir = Path(here('data/marker_database/harmonized'))

In [3]:
# Marker genes
markers = {
    "beta": ["INS", "IAPP", "G6PC2", "GLIS3"], # https://www.science.org/doi/10.1126/sciadv.ady0080
    "alpha": ["GCG", "ARX", "MAFB"],
    "delta": ["SST", "HHEX", "RBP4"],
    "gamma": ["PPY", "ARX", "ETV1"],
    "epsilon": ["GHRL", "AFF3", "THSD4", "PHGR1"], # https://www.biorxiv.org/content/10.1101/2025.10.01.679230v1.full.pdf
    "acinar": ["PRSS1", "PRSS2", "AMY2B"],
    "acivated_stellate": ["COL1A1", "COL3A1", "MMP2"], # https://www.frontiersin.org/journals/endocrinology/articles/10.3389/fendo.2024.1532609/full # https://pubmed.ncbi.nlm.nih.gov/30466735/
    "quiescent_stellate": ["RGS5", "FABP4", "ADIRF"], # https://portal.findresearcher.sdu.dk/en/publications/identification-of-markers-for-quiescent-pancreatic-stellate-cells/ # https://apc.amegroups.org/article/view/4871/html
    "endothelial": ["PECAM1", "PLVAP"],
    "schawnn": ["SOX10", "S100B", "CRYAB"], # https://www.pnas.org/doi/full/10.1073/pnas.1612136114
    "cycling": ["TOP2A", "MKI67"],
    "ductal": ["KRT19", "SOX9", "HNF1B"], # https://pmc.ncbi.nlm.nih.gov/articles/PMC3945019/ # https://pmc.ncbi.nlm.nih.gov/articles/PMC3225990/
    "ducal_muc5": ["MUC5B", "ERN2", "CYP2C18", "MYO7B", "DMBT1", "TFF1", "TFF2", "TFF3"],  # https://www.science.org/doi/10.1126/sciadv.ady0080 # https://www.gastrojournal.org/article/S0016-5085(20)35399-3/fulltext?referrer=https%3A%2F%2Fpubmed.ncbi.nlm.nih.gov%2F
    "mast": ["TPSAB1", "TPSB2", "CPA3", "KIT"], #https://www.nature.com/articles/s41388-025-03590-y
    "macrophages": ["CSF1R", "CD68"], # https://pmc.ncbi.nlm.nih.gov/articles/PMC11851012/ #https://www.nature.com/articles/labinvest2016116
    "B-cells": ["CD53"], # https://pmc.ncbi.nlm.nih.gov/articles/PMC6926393/
    "hematopoietic_cells": ["PTPRC"],
    "antigen_presenting": ["CD68", "HLA-DRA"]} 

Load data

In [4]:
# Adata object
adata = ad.read_h5ad(os.path.join(anndata_dir, "AG_combined.h5ad"))

Find clusters

In [ ]:
# default
sc.tl.leiden(adata, n_iterations=-1, flavor = 'igraph', key_added = 'leiden', random_state= 1000, resolution = 1)

In [ ]:
# higher resolution
sc.tl.leiden(adata, n_iterations=-1, flavor = 'igraph', key_added = 'leiden_1.5', random_state= 1000, resolution = 1.5)

Save Adata object

In [ ]:
adata.write_h5ad(os.path.join(anndata_dir, "AG_combined.h5ad"))

Save clusterings

In [ ]:
clustering = adata.obs.loc[:,adata.obs.columns.str.startswith('leiden_1.5')].copy()
clustering.to_csv(os.path.join(files_dir, 'leiden_1.5_clusterings_igraph.csv'), index_label='barcode')
clustering = adata.obs.loc[:,adata.obs.columns.str.startswith('leiden')].copy()
clustering.to_csv(os.path.join(files_dir, 'leiden_clusterings_igraph.csv'), index_label='barcode')

See clusters

In [ ]:
sc.pl.umap(adata, color = "leiden_1.5", legend_loc= "on data", legend_fontsize= 5)

In [ ]:
sc.tl.dendrogram(adata, groupby="leiden_1.5", use_rep='X_latent_1')

In [ ]:
sc.pl.dotplot(adata, markers, groupby = 'leiden_1.5', dendrogram= True, log = True)

# Diffentiel expressed genes

### Set up 

In [ ]:
cluster_key = 'leiden_1.5'
sample_key = 'ic_id_platform_adjusted_sample'
design= '~ ic_id_platform_adjusted_sample + leiden_1.5'
dds_name = 'leiden_1_5_dds.pkl'

### Percentage of cells expressing genes

In [ ]:
pct_genes_df = dg.compute_pct_expressing(ad = adata, cluster_key = cluster_key, layer="counts", workers=60)

### Prepare DDS object

In [ ]:
dds = dg.prepare_pseudobulk_deseq_analysis(ad = adata, 
                                           n_cells = 25,
                                           sample_key = sample_key, 
                                           cluster_key = cluster_key, 
                                           design = design, 
                                           layer='counts', 
                                           func='sum', 
                                           workers=60)

### Find differential genes between two clusters

In [ ]:
# Define comparisons
comparisons = [("22", "23"),
               ("22", "21"),
               ("22", "20"),
               ("16", "21"),
               ("16", "20"),
               ("20", "21"),
               ("17", "18")]

# Wald test
results_list = [
    dg.diff_genes_two_clusters(dds_obj = dds, cluster_index=cluster_key,
                            cluster_1=c1, cluster_2=c2, workers=60).assign(comparison=f"{c1}_vs_{c2}")
    for c1, c2 in comparisons
]

### Extract and set up results

In [ ]:
# Merge all results
all_results = pd.concat(results_list, ignore_index=False)

# Merge percentage of genes expressed
all_results = all_results.merge(pct_genes_df, left_index=True, right_index=True, how="left")

# Convert comparisons to categorical (optional, for sorting later)
comparison_order = [f"{c1}_vs_{c2}" for c1, c2 in comparisons]
all_results['comparison'] = pd.Categorical(all_results['comparison'], categories=comparison_order, ordered=True)

# Extract c1 from the comparison string
all_results['c1'] = all_results['comparison'].str.split('_vs_').str[0]

# Add helper column for percent expressed in c1
all_results['pct_expr_c1'] = all_results.apply(
    lambda row: row[f"pct_expr_cluster_{row['c1']}"], axis=1
)

# Reorder columns
cols_to_move = ['comparison', 'pct_expr_c1', ]
other_cols = [c for c in all_results.columns if c not in cols_to_move]
all_results = all_results[cols_to_move + other_cols]

### Filter results

In [ ]:
# Filter results
all_results_flt = all_results.query('padj <= 0.05 and log2FoldChange >= 1')

In [ ]:
comparisons

In [ ]:
top10 = (
    all_results_flt
    .query("comparison == '16_vs_21'")
    .nlargest(10, ['log2FoldChange', 'pct_expr_c1'])
)

print(top10)

### Save results

In [ ]:
all_results.to_csv(os.path.join(files_dir, "deg_wald_all_22_23_21_20_16_17_18.csv"), index_label= "gene_symbol")
all_results_flt.to_csv(os.path.join(files_dir, "deg_wald_sig_22_23_21_20_16_17_18.csv"), index_label= "gene_symbol")

## Module score of subtypes of hepatic stellate cells from [Merens, Vincent et al.](https://doi.org/10.1016/j.jhepr.2024.101223)

(Access: 08-12-2025)

In [18]:
genes_hsc = pd.read_csv(os.path.join(files_dir, "mmc5_SF2_HSC_subtype_genesets.csv"))

### Convert mouse gene symbols to human and create dictionary

In [19]:
# mouse to human symbol
genes_hsc = mi.mouse_2_human(genes_hsc, mouse_col = "Gene")

# dictionary
gene_hsc_list = {}

df_grouped = genes_hsc.groupby('HSC stage')['human_symbol'].apply(list)
for celltype, genes in df_grouped.items(): 
    key = f"{celltype}"
    gene_hsc_list[key] = genes

### UCell score

In [20]:
uc.compute_ucell_scores(adata, signatures=gene_hsc_list, n_jobs= 60)
adata.obs[adata.obs.columns[adata.obs.columns.str.endswith('UCell')]].to_csv(os.path.join(files_dir, "hcs_marker_gene_scores.csv"), index_label='barcode')

Differential expressed genes between cluster 22 and 23

In [ ]:
cell_counts = adata.obs.groupby(['ic_id_platform_adjusted_sample', 'leiden_1.5'], observed = True).size()
cell_counts_df = cell_counts.reset_index(name='n_cells')
print(cell_counts_df)

In [ ]:
n_cells = 25  # number of cells per group
# create a new list to collect indices of selected cells
selected_indices = []

# group by sample and cluster
for sample_cluster, indices in adata.obs.groupby(['ic_id_platform_adjusted_sample', 'leiden_1.5']).indices.items():
    if len(indices) < n_cells:
        # skip this group entirely
        continue
    elif len(indices) == n_cells:
        # take all if exactly n_cells
        selected_indices.extend(indices)
    else:
        # randomly sample n_cells
        selected_indices.extend(np.random.choice(indices, n_cells, replace=False))

# subset the AnnData
adata_subsampled = adata[selected_indices].copy()

In [ ]:
cell_counts = adata_subsampled.obs.groupby(['ic_id_platform_adjusted_sample', 'leiden_1.5'], observed = True).size()
cell_counts_df = cell_counts.reset_index(name='n_cells')
print(cell_counts_df)

In [ ]:
pseudobulk_adata = sc.get.aggregate(adata = adata_subsampled,
                           by = ['ic_id_platform_adjusted_sample', 'leiden_1.5'], 
                                    layer = 'counts', func = 'sum')

In [ ]:
from pydeseq2.dds import DeseqDataSet
from pydeseq2.default_inference import DefaultInference
from pydeseq2.ds import DeseqStats

counts_df = pd.DataFrame(pseudobulk_adata.layers['sum'], columns=pseudobulk_adata.var_names, index=pseudobulk_adata.obs_names)
metadata_df = pseudobulk_adata.obs[['leiden_1.5', 'ic_id_platform_adjusted_sample']].copy()
metadata_df = metadata_df.set_index(counts_df.index)
genes_to_keep = counts_df.columns[counts_df.sum(axis=0) >= 10]
counts_df = counts_df[genes_to_keep]

# Create Deseq object
inference = DefaultInference(n_cpus=60)

dds = DeseqDataSet(
    counts=counts_df,
    metadata=metadata_df,
    design="~ ic_id_platform_adjusted_sample + leiden_1.5", 
    inference=inference,
)

# Fit dispertion and logfoldchanges
dds.deseq2()

# save dds
import pickle as pkl
with open(os.path.join(files_dir, "leiden_1_5_dds.pkl"), "wb") as f:
    pkl.dump(dds, f)

### Differential expressed genes between cluster 22 and 23

In [ ]:
ds = DeseqStats(
    dds,
    contrast=("leiden_1.5", "22", "23"), 
    inference = inference)
ds.run_wald_test()
ds.summary()

Save object

In [ ]:
# Save object
import pickle as pkl
with open(os.path.join(files_dir, "leiden_15_22_vs_23.pkl"), "wb") as f:
    pkl.dump(ds, f)

Extract results and add how many % of cells in cluster 22 express genes

In [ ]:
res = ds.results_df.copy()
cluster_col = "leiden_1.5"
cluster = "22"  # example cluster you ran DE for

# If no genes pass, exit early
if res.shape[0] > 0:
    # subset AnnData to this cluster
    adata_c = adata[adata.obs[cluster_col] == cluster]

    # get only genes in "res"
    gene_subset = res.index
    adata_c = adata_c[:, gene_subset]

    # extract counts
    X = adata_c.layers['counts']
    if hasattr(X, "toarray"):
        X = X.toarray()

    # % cells expressing
    pct_expr = (X > 0).sum(axis=0) / X.shape[0] * 100
    pct_expr = pd.Series(
        np.round(pct_expr, 2),
        index=gene_subset,
        name=f"pct_expr_cluster_{cluster}"
    )

    # merge into res results
    res = res.merge(pct_expr, left_index=True, right_index=True, how="left")

# sort according results
filtered_sorted = res[(res['padj'] <= 0.05) & (res['log2FoldChange'] > 1) & (res['pct_expr_cluster_22'] > 50)].sort_values(
    by=['log2FoldChange', 'padj'],
    ascending=[False, True])

### Save results

In [ ]:
res.to_csv(os.path.join(files_dir, "de_results_22_vs_23.csv"), index_label= "gene_symbol")

### Differential expressed genes between cluster 22 and 21

In [ ]:
# Run test
ds_21 = DeseqStats(
    dds,
    contrast=("leiden_1.5", "22", "21"), 
    inference = inference)
ds_21.run_wald_test()
ds_21.summary()

# Save objects
with open(os.path.join(files_dir, "leiden_15_22_vs_21.pkl"), "wb") as f:
    pkl.dump(ds_21, f)

# Extract results
res = ds_21.results_df.copy()
cluster_col = "leiden_1.5"
cluster = "22"  # example cluster you ran DE for

# If no genes pass, exit early
if res.shape[0] > 0:
    # subset AnnData to this cluster
    adata_c = adata[adata.obs[cluster_col] == cluster]

    # get only genes in "res"
    gene_subset = res.index
    adata_c = adata_c[:, gene_subset]

    # extract counts
    X = adata_c.layers['counts']
    if hasattr(X, "toarray"):
        X = X.toarray()

    # % cells expressing
    pct_expr = (X > 0).sum(axis=0) / X.shape[0] * 100
    pct_expr = pd.Series(
        np.round(pct_expr, 2),
        index=gene_subset,
        name=f"pct_expr_cluster_{cluster}"
    )

    # merge into res results
    res = res.merge(pct_expr, left_index=True, right_index=True, how="left")

# Save results
res.to_csv(os.path.join(files_dir, "de_results_22_vs_21.csv"), index_label= "gene_symbol")

# sort according results
filtered_sorted = res[(res['padj'] <= 0.05) & (res['log2FoldChange'] > 1) & (res['pct_expr_cluster_22'] > 50)].sort_values(
    by=['log2FoldChange', 'padj'],
    ascending=[False, True])

In [ ]:
filtered_sorted = res[(res['padj'] <= 0.05) & (res['log2FoldChange'] > 1) & (res['pct_expr_cluster_22'] > 50)].sort_values(
    by=['log2FoldChange', 'padj'],
    ascending=[False, True])

In [ ]:
filtered_sorted

In [ ]:
filtered_sorted = res[(res['padj'] <= 0.05) & (res['log2FoldChange'] > 1) & (res['pct_expr_cluster_22'] > 10)].sort_values(
    by=['padj'],
    ascending=[True])
sc.pl.dotplot(adata, list(filtered_sorted.head(50).index), groupby = 'leiden_1.5', dendrogram= True, log = True)

### Differential expressed genes between cluster 20 and 21

In [ ]:
# Run test
ds_20_21 = DeseqStats(
    dds,
    contrast=("leiden_1.5", "20", "21"), 
    inference = inference)
ds_20_21.run_wald_test()
ds_20_21.summary()

# Save objects
with open(os.path.join(files_dir, "leiden_15_20_vs_21.pkl"), "wb") as f:
    pkl.dump(ds_20_21, f)

# Extract results
res = ds_20_21.results_df.copy()
cluster_col = "leiden_1.5"
cluster = "20"  # example cluster you ran DE for

# If no genes pass, exit early
if res.shape[0] > 0:
    # subset AnnData to this cluster
    adata_c = adata[adata.obs[cluster_col] == cluster]

    # get only genes in "res"
    gene_subset = res.index
    adata_c = adata_c[:, gene_subset]

    # extract counts
    X = adata_c.layers['counts']
    if hasattr(X, "toarray"):
        X = X.toarray()

    # % cells expressing
    pct_expr = (X > 0).sum(axis=0) / X.shape[0] * 100
    pct_expr = pd.Series(
        np.round(pct_expr, 2),
        index=gene_subset,
        name=f"pct_expr_cluster_{cluster}"
    )

    # merge into res results
    res = res.merge(pct_expr, left_index=True, right_index=True, how="left")

# Save results
res.to_csv(os.path.join(files_dir, "de_results_20_vs_21.csv"), index_label= "gene_symbol")

# sort according results
filtered_sorted = res[(res['padj'] <= 0.05) & (res['log2FoldChange'] > 1) & (res['pct_expr_cluster_20'] > 50)].sort_values(
    by=['log2FoldChange', 'padj'],
    ascending=[False, True])

In [ ]:
filtered_sorted.head(20)

## Differential expressed genes between cluster 21 and 20

In [ ]:
# Run test
ds_21_20 = DeseqStats(
    dds,
    contrast=("leiden_1.5", "21", "20"), 
    inference = inference)
ds_21_20.run_wald_test()
ds_21_20.summary()

# Save objects
with open(os.path.join(files_dir, "leiden_15_21_vs_20.pkl"), "wb") as f:
    pkl.dump(ds_21_20, f)

# Extract results
res = ds_21_20.results_df.copy()
cluster_col = "leiden_1.5"
cluster = "21"  # example cluster you ran DE for

# If no genes pass, exit early
if res.shape[0] > 0:
    # subset AnnData to this cluster
    adata_c = adata[adata.obs[cluster_col] == cluster]

    # get only genes in "res"
    gene_subset = res.index
    adata_c = adata_c[:, gene_subset]

    # extract counts
    X = adata_c.layers['counts']
    if hasattr(X, "toarray"):
        X = X.toarray()

    # % cells expressing
    pct_expr = (X > 0).sum(axis=0) / X.shape[0] * 100
    pct_expr = pd.Series(
        np.round(pct_expr, 2),
        index=gene_subset,
        name=f"pct_expr_cluster_{cluster}"
    )

    # merge into res results
    res = res.merge(pct_expr, left_index=True, right_index=True, how="left")

# Save results
res.to_csv(os.path.join(files_dir, "de_results_21_vs_20.csv"), index_label= "gene_symbol")

# sort according results
filtered_sorted = res[(res['padj'] <= 0.05) & (res['log2FoldChange'] > 1) & (res['pct_expr_cluster_21'] > 50)].sort_values(
    by=['log2FoldChange', 'padj'],
    ascending=[False, True])

# show top 20
filtered_sorted.head(20)

### Differential expressed genes between cluster 16 vs 21

In [ ]:
# Run test
ds_16_21 = DeseqStats(
    dds,
    contrast=("leiden_1.5", "16", "21"), 
    inference = inference)
ds_16_21.run_wald_test()
ds_16_21.summary()

# Save objects
with open(os.path.join(files_dir, "leiden_15_16_vs_21.pkl"), "wb") as f:
    pkl.dump(ds_16_21, f)

# Extract results
res = ds_16_21.results_df.copy()
cluster_col = "leiden_1.5"
cluster = "16"  # example cluster you ran DE for

# If no genes pass, exit early
if res.shape[0] > 0:
    # subset AnnData to this cluster
    adata_c = adata[adata.obs[cluster_col] == cluster]

    # get only genes in "res"
    gene_subset = res.index
    adata_c = adata_c[:, gene_subset]

    # extract counts
    X = adata_c.layers['counts']
    if hasattr(X, "toarray"):
        X = X.toarray()

    # % cells expressing
    pct_expr = (X > 0).sum(axis=0) / X.shape[0] * 100
    pct_expr = pd.Series(
        np.round(pct_expr, 2),
        index=gene_subset,
        name=f"pct_expr_cluster_{cluster}"
    )

    # merge into res results
    res = res.merge(pct_expr, left_index=True, right_index=True, how="left")

# Save results
res.to_csv(os.path.join(files_dir, "de_results_16_vs_21.csv"), index_label= "gene_symbol")

# sort according results
filtered_sorted = res[(res['padj'] <= 0.05) & (res['log2FoldChange'] > 1) & (res['pct_expr_cluster_16'] > 50)].sort_values(
    by=['log2FoldChange', 'padj'],
    ascending=[False, True])

# show top 20
filtered_sorted.head(20)

In [ ]:
filtered_sorted = res[(res['padj'] <= 0.05) & (res['log2FoldChange'] > 1) & (res['pct_expr_cluster_16'] > 50)].sort_values(
    by=['log2FoldChange', 'padj', 'pct_expr_cluster_16'],
    ascending=[False, True, False])

In [ ]:
sc.pl.dotplot(adata, list(filtered_sorted.head(50).index), groupby = 'leiden_1.5', dendrogram= True, log = True)

Get number of samples represented in cluster 22 in the pseudobulk

In [ ]:
samples_in_22 = list(metadata_df[metadata_df['leiden_1.5'] == "22"]['ic_id_platform_adjusted_sample'])
test = adata.obs[["ic_id_study", "name", "ic_id_platform_adjusted_sample", "library_prep"]].reset_index()
test_22 = test[test["ic_id_platform_adjusted_sample"].isin(samples_in_22)]
test_22[["ic_id_study", "name", "ic_id_platform_adjusted_sample", "library_prep"]].drop_duplicates()

Studies in general represented in cluster 22 (but there were not enough cells to include them)

In [ ]:
test = adata.obs[["ic_id_study", "name", "ic_id_platform_adjusted_sample", "library_prep", "leiden_1.5"]].reset_index()
test_22 = test[test["leiden_1.5"].isin(['22'])]
test_22[["ic_id_study", "name", "leiden_1.5", "library_prep"]].drop_duplicates()

In [ ]:
test = adata.obs[["ic_id_study", "name", "ic_id_platform_adjusted_sample"]].reset_index()
test[["ic_id_study", "name", "ic_id_platform_adjusted_sample"]].drop_duplicates()

In [ ]:
# sc.pl.dotplot(adata, ["FOSL1", "EGR3", "NFKB2", "CLEC4G", "LYVE1", "STAB2", 
#                      "ELN", "FBLN2", "ENTPD2", "COL15A1", "CD34"], groupby = 'leiden_1.5', dendrogram= True, log = True)

sc.pl.dotplot(adata, ["CCL2", "TPM1", "THBS1", "TIMP1"], groupby = 'leiden_1.5', dendrogram= True, log = True)


In [ ]:
sc.pl.dotplot(adata, list(filtered_sorted.head(20).index), groupby = 'leiden_1.5', dendrogram= True, log = True)

In [ ]:
# create a dictionary to map cluster to annotation label
cluster2annotation = {
    "0": "alpha",
    "1": "gamma",
    "2": "alpha",
    "3": "alpha",
    "4": "alpha",
    "5": "alpha",
    "6": "cycling",
    "7": "alpha",
    "8": "beta",
    "9": "beta",
    "10": "beta",
    "11": "beta",
    "12": "delta",
    "13": "acinar",
    "14": "quiescent_stellate",
    "15": "activated_stellate",
    "16": "antigen_presenting",
    "17": "endothelial",
    "18": "ductal",
    "19": "ductal_muc5",
    "20": "epsilon",
    "21": "mast",
    "22": "schwann"
}

# add a new `.obs` column called `cell type` by mapping clusters to annotation using pandas `map` function
adata.obs["manual_annotation"] = adata.obs["leiden"].map(cluster2annotation).astype("category")

In [ ]:
# Fix marker dictionary keys (typos corrected)
markers = {
    "beta": ["INS", "IAPP", "G6PC2"],
    "alpha": ["GCG", "ARX", "MAFB"],
    "delta": ["SST", "HHEX", "RBP4"],
    "gamma": ["PPY", "ARX", "ETV1"],
    "epsilon": ["GHRL"],
    "acinar": ["PRSS1"],
    "activated_stellate": ["COL1A1"],        # fixed spelling
    "quiescent_stellate": ["RGS5"],
    "endothelial": ["PECAM1", "PLVAP"],
    "schwann": ["SOX10", "S100B"],           # fixed spelling
    "cycling": ["TOP2A", "MKI67"],
    "ductal": ["KRT19"],
    "ductal_muc5": ["MUC5B", "ERN2", "CYP2C18", "MYO7B", "DMBT1"],  # fixed spelling
    "mast": ["TPSAB1", "TPSB2", "CPA3", "KIT"],
    "antigen_presenting": ["PTPRC", "CD68", "HLA-DRA", "CD53", "CD2"],
}

# Set annotation column
adata.obs["manual_annotation"] = (
    adata.obs["leiden"].map(cluster2annotation).astype("category")
)

sc.tl.dendrogram(adata, groupby="manual_annotation", use_rep="X_latent_1")

# Extract dendrogram-ordered groups
ordered_groups = adata.uns["dendrogram_manual_annotation"]["categories_ordered"]

# Reorder marker dict keys to match this group order
markers_ordered = {grp: markers[grp] for grp in ordered_groups if grp in markers}

# Plot with dendrogram
sc.pl.dotplot(
    adata,
    markers_ordered,
    groupby="manual_annotation",
    dendrogram=True,
    log=True
)


In [ ]:
cell_counts_per_sample = adata.obs.groupby('ic_id_platform_adjusted_sample', observed=False).size()
low_cell_samples = cell_counts_per_sample[cell_counts_per_sample < 20].index  # example threshold
adata_sub = adata[~adata.obs['ic_id_platform_adjusted_sample'].isin(low_cell_samples)]

In [ ]:
pseudobulk_adata = sc.get.aggregate(adata = adata_sub,
                           by = ['ic_id_platform_adjusted_sample', 'leiden'], 
                                    layer = 'counts', func = 'sum')

In [ ]:
from pydeseq2.dds import DeseqDataSet
from pydeseq2.default_inference import DefaultInference
from pydeseq2.ds import DeseqStats

counts_df = pd.DataFrame(pseudobulk_adata.layers['sum'], columns=pseudobulk_adata.var_names, index=pseudobulk_adata.obs_names)
metadata_df = pseudobulk_adata.obs[['leiden', 'ic_id_platform_adjusted_sample']].copy()
metadata_df = metadata_df.set_index(counts_df.index)
genes_to_keep = counts_df.columns[counts_df.sum(axis=0) >= 10]
counts_df = counts_df[genes_to_keep]

# Create Deseq object
inference = DefaultInference(n_cpus=30)

dds = DeseqDataSet(
    counts=counts_df,
    metadata=metadata_df,
    design="~ ic_id_platform_adjusted_sample + leiden", 
    refit_cooks=True,
    inference=inference,
)

# Fit dispertion and logfoldchanges
dds.deseq2()

import pickle as pkl
with open(os.path.join(file_path, "leiden_dds.pkl"), "wb") as f:
    pkl.dump(dds, f)

In [ ]:
genes_to_keep = counts_df.columns[counts_df.sum(axis=0) >= 10]
counts_df = counts_df[genes_to_keep]

In [ ]:
# Create Deseq object
inference = DefaultInference(n_cpus=30)

dds = DeseqDataSet(
    counts=counts_df,
    metadata=metadata_df,
    design="~ ic_id_platform_adjusted_sample + leiden", 
    refit_cooks=True,
    inference=inference,
)

# Fit dispertion and logfoldchanges
dds.deseq2()

In [ ]:
import pickle as pkl
with open(os.path.join(file_path, "leiden_dds.pkl"), "wb") as f:
    pkl.dump(dds, f)

In [ ]:
ds = DeseqStats(
    dds,
    contrast=("leiden", "19", "18"), 
    inference = inference)
ds.run_wald_test()
ds.summary()

In [ ]:
# azimuth markers
azi_markers = pd.read_csv(os.path.join(harmo_dir, 'azimuth_genes.csv'), sep=",", dtype=str) 
# Remove genes that are not in var names
azi_markers = azi_markers[azi_markers['gene'].isin(adata.var_names)]
# make to dictionary
azi_markers = azi_markers.groupby('cell_type')['gene'].apply(list).apply(list).to_dict()

In [ ]:
# pangola markers
pan_markers = pd.read_csv(os.path.join(harmo_dir, 'panglao_genes.csv'), sep=",", dtype=str) 
# Remove genes that are not in var names
pan_markers = pan_markers[pan_markers['gene'].isin(adata.var_names)]
# make to dictionary
pan_markers = pan_markers.groupby('cell_type')['gene'].apply(list).apply(list).to_dict()

In [ ]:
# cellmarker markers
cell_markers = pd.read_csv(os.path.join(harmo_dir, 'cellmarker_genes.csv'), sep=",", dtype=str) 
# Remove genes that are not in var names
cell_markers = cell_markers[cell_markers['gene'].isin(adata.var_names)]
# make to dictionary
cell_markers = cell_markers.groupby('cell_type')['gene'].apply(list).apply(list).to_dict()

Find clusters (run until optimal clustering is found)

Save object with leiden

In [ ]:
adata.write(os.path.join(anndata_dir, "AG_combined.h5ad"))

Marker gene expression

In [ ]:
mi.set_my_theme()
fig = sc.pl.dotplot(adata, azi_markers, groupby = 'leiden', return_fig= True, show= False, figsize= (31,5))
fig.savefig(os.path.join(plot_dir, "manual_annotation_azimuth_dotplot.pdf"), bbox_inches="tight")

In [ ]:
mi.set_my_theme()
fig = sc.pl.dotplot(adata, pan_markers, groupby = 'leiden', return_fig= True, show= False, figsize= (50,5), log = True)
fig.savefig(os.path.join(plot_dir, "manual_annotation_panglao_dotplot.pdf"), bbox_inches="tight")

In [ ]:
mi.set_my_theme()
fig = sc.pl.dotplot(adata, cell_markers, groupby = 'leiden', return_fig= True, show= False, figsize= (50,5), log = True)
fig.savefig(os.path.join(plot_dir, "manual_annotation_cellmarker_dotplot.pdf"), bbox_inches="tight")

Do manual annotation based on marker genes expressions

In [ ]:
# create a dictionary to map cluster to annotation label
cluster2annotation = {
    "0": "alpha",
    "1": "gamma",
    "2": "alpha",
    "3": "alpha",
    "4": "alpha",
    "5": "alpha",
    "6": "cycling",
    "7": "alpha",
    "8": "beta",
    "9": "beta",
    "10": "beta",
    "11": "beta",
    "12": "delta",
    "13": "acinar",
    "14": "quiescent_stellate_cell",
    "15": "activated_stellate_cell",
    "16": "macrophage",
    "17": "endothelial",
    "18": "ductal",
    "19": "ductal",
    "20": "epsilon",
    "21": "mast_cell",
    "22": "schwann_cell"
}

# add a new `.obs` column called `cell type` by mapping clusters to annotation using pandas `map` function
adata.obs["manual_annotation"] = adata.obs["leiden"].map(cluster2annotation).astype("category")

In [ ]:
sc.pl.umap(adata, color = "MUC5B", legend_loc= "on data", legend_fontsize= 5)

Save object and leiden clusters and manual annotation

In [ ]:
clustering = adata.obs.loc[:,adata.obs.columns.str.startswith('leiden')].copy()
clustering.to_csv(os.path.join(files_dir, 'leiden_clusterings_igraph.csv'), index_label='barcode')

In [ ]:
manual_anno = adata.obs.loc[:,adata.obs.columns.str.startswith('manual_annotation')].copy()
manual_anno.to_csv(os.path.join(files_dir, 'manual_annotation.csv'), index_label='barcode')

Scale expression of marker genes

In [ ]:
# marker genes
# path to marker genes
genelists_path = glob.glob(os.path.join(harmo_dir, '*.csv'))

genelists = {}

for file in genelists_path:
    name = Path(file).stem.replace('_genes', '')
    df = pd.read_csv(file, sep=",", dtype=str)
    df_grouped = df.groupby('cell_type')['gene'].apply(list)
    for celltype, genes in df_grouped.items():
        key = f"{name}_{celltype}"
        genelists[key] = genes

genes_to_scale = []

for _, genes in genelists.items():
    genes_to_scale.extend(genes)

# get unique
genes_to_scale = list(set(genes_to_scale))

In [ ]:
idx = adata.var_names.isin(genes_to_scale)
adata_sub = adata[:, idx].copy()
scaled_subset = sc.pp.scale(adata_sub, copy=True).X
adata_sub.layers["scaled"] = scaled_subset

In [ ]:
sc.pl.matrixplot(
    adata_sub,
    cell_markers,
    "leiden",
    colorbar_title="mean z-score",
    layer="scaled", 
    vmin=-2,
    vmax=2,
    cmap="RdBu_r",
)

In [ ]:
pan_markers.keys()